# 03 - Multi-Candidate Station Analysis

This notebook finds the best public transport stations for each employee and the office by evaluating multiple candidate stations instead of just the nearest one. We find 5 nearest stations for each location, then evaluate all combinations to find the optimal route.

In [ ]:
# Import the libraries we need for station analysis
import pandas as pd
import numpy as np
import requests
import time
from dotenv import load_dotenv
import os
from sklearn.neighbors import BallTree
import geopandas as gpd
from shapely.geometry import Point

In [ ]:
# Load environment variables and get API key
load_dotenv()

# Get OpenRouteService API key for routing
ORS_API_KEY = os.getenv('OPENROUTESERVICE_API_KEY')

# Load J&J Office snapped coordinates from the geocoding notebook output
import json
with open('data/data/jj_office_snapped.json', 'r') as f:
    jj_office_data = json.load(f)

JJ_OFFICE = {
    'address': jj_office_data['address'],
    'latitude': jj_office_data['original_latitude'],
    'longitude': jj_office_data['original_longitude'],
    'snapped_latitude': jj_office_data['snapped_latitude'],
    'snapped_longitude': jj_office_data['snapped_longitude']
}

print(f"Loaded J&J office coordinates from 02_geocoding output")
print(f"Address: {JJ_OFFICE['address']}")
print(f"Original: ({JJ_OFFICE['latitude']}, {JJ_OFFICE['longitude']})")
print(f"Snapped: ({JJ_OFFICE['snapped_latitude']}, {JJ_OFFICE['snapped_longitude']})")

In [ ]:
# Load the geocoded employee data from the previous notebook
print(f"Loaded {len(employee_df)} employees with geocoded coordinates")

In [4]:
# Optional: Display employee data for verification

In [ ]:
# Function to fetch all public transport stops (trains + buses) in Hamburg metropolitan area from OpenStreetMap
def fetch_hvv_stations():
    """Fetch all public transport stops in Hamburg area using Overpass API"""

    overpass_url = "https://overpass-api.de/api/interpreter"

    # Comprehensive query for all public transport types
    overpass_query = """
    [out:json][timeout:180];
    (
      node["railway"="station"](53.4,9.6,53.8,10.3);
      node["railway"="halt"](53.4,9.6,53.8,10.3);
      node["station"="subway"](53.4,9.6,53.8,10.3);
      node["station"="light_rail"](53.4,9.6,53.8,10.3);
      
      node["highway"="bus_stop"](53.4,9.6,53.8,10.3);
      node["public_transport"="stop_position"](53.4,9.6,53.8,10.3);
      node["public_transport"="platform"](53.4,9.6,53.8,10.3);
      
      way["railway"="station"](53.4,9.6,53.8,10.3);
      way["railway"="halt"](53.4,9.6,53.8,10.3);
      
      relation["railway"="station"](53.4,9.6,53.8,10.3);
      relation["railway"="halt"](53.4,9.6,53.8,10.3);
    );
    out center;
    """

    try:
        response = requests.post(
            overpass_url,
            data=overpass_query,
            headers={
                "User-Agent": "Python requests"
            },
            timeout=240,  # Increased timeout for comprehensive query
        )

        print("Status code:", response.status_code)
        response.raise_for_status()

        data = response.json()

        stations = []

        for element in data.get("elements", []):
            tags = element.get("tags", {})

            # Nodes have lat/lon
            if "lat" in element:
                lat = element["lat"]
                lon = element["lon"]
            # Ways/relations have center
            elif "center" in element:
                lat = element["center"]["lat"]
                lon = element["center"]["lon"]
            else:
                continue

            # Determine transport type
            transport_type = "unknown"
            if tags.get("railway"):
                transport_type = "rail"
            elif tags.get("highway") == "bus_stop":
                transport_type = "bus"
            elif tags.get("public_transport"):
                transport_type = tags.get("public_transport")
            elif tags.get("station"):
                transport_type = tags.get("station")

            stations.append({
                "station_id": element["id"],
                "name": tags.get("name", "Unknown"),
                "latitude": lat,
                "longitude": lon,
                "station_type": tags.get("railway", tags.get("highway", tags.get("public_transport", ""))),
                "transport_type": transport_type,
                "public_transport": tags.get("public_transport", ""),
            })

        return pd.DataFrame(stations)

    except requests.exceptions.HTTPError as e:
        print("HTTP Error:", e)
        print(response.text[:1000])
        return pd.DataFrame()

    except Exception as e:
        print("Error:", e)
        return pd.DataFrame()


# Fetch all public transport stops in Hamburg area
hvv_stations = fetch_hvv_stations()

print(f"\nFound {len(hvv_stations)} public transport stops")

# Display sample data
print(hvv_stations.head(10))

# Save stations to CSV
hvv_stations.to_csv("hvv_stations.csv", index=False)

# Show statistics by transport type
print(f"\nTransport types:")
print(hvv_stations['transport_type'].value_counts())

print(f"\nStation types:")
print(hvv_stations['station_type'].value_counts())

print(f"\nGeographic coverage:")
print(f"Latitude range: {hvv_stations['latitude'].min():.4f} to {hvv_stations['latitude'].max():.4f}")
print(f"Longitude range: {hvv_stations['longitude'].min():.4f} to {hvv_stations['longitude'].max():.4f}")

In [6]:
# Optional: Display stations data for verification

In [ ]:
# Function to remove duplicate stations (some stations appear multiple times in OSM data)
def deduplicate_stations(stations_df):
    """Remove exact duplicate stations from the dataset"""

    stations_df = stations_df.copy()

    # Remove duplicate OSM objects (same station_id)
    stations_df = stations_df.drop_duplicates(subset=["station_id"])

    # Remove exact duplicate stop records (same name, lat, lon)
    stations_df = stations_df.drop_duplicates(
        subset=["name", "latitude", "longitude"]
    )

    return stations_df


# Apply deduplication to our station data
hvv_stations = deduplicate_stations(hvv_stations)

print(f"Stations after deduplication: {len(hvv_stations)}")

In [ ]:
# Analyze and categorize the transport types to create meaningful categories
print("=== Current Transport Types ===")
print(hvv_stations['transport_type'].value_counts())

# Function to create meaningful transport categories from the messy OSM data
def categorize_transport(row):
    """Create meaningful transport categories from raw OSM tags"""
    
    # Check railway first (higher priority for rail stations)
    if row['station_type'] in ['station', 'halt']:
        return 'rail'
    
    # Check for bus stops
    if row['station_type'] == 'bus_stop':
        return 'bus'
    
    # Check transport_type field
    transport = row['transport_type']
    if transport == 'rail':
        return 'rail'
    elif transport == 'bus':
        return 'bus'
    elif transport in ['subway', 'light_rail', 'tram']:
        return 'urban_rail'
    
    # Default based on public_transport tag
    pt = row['public_transport']
    if pt == 'station':
        return 'rail'
    elif pt == 'stop_position':
        return 'bus'  # Most stop_positions are bus stops
    
    return 'other'

# Apply categorization to all stations
hvv_stations['transport_category'] = hvv_stations.apply(categorize_transport, axis=1)

print("\n=== Categorized Transport Types ===")
print(hvv_stations['transport_category'].value_counts())

# Filter to keep only relevant categories (bus, rail, urban_rail)
hvv_stations_filtered = hvv_stations[hvv_stations['transport_category'].isin(['bus', 'rail', 'urban_rail'])]

print(f"\nStations after filtering: {len(hvv_stations_filtered)}")
print(f"Stations by category:")
print(hvv_stations_filtered['transport_category'].value_counts())

# Save filtered stations
hvv_stations_filtered.to_csv('data/hvv_stations_filtered.csv', index=False)
print("Saved filtered stations to data/hvv_stations_filtered.csv")

In [9]:
# Skip - This is replaced by the filtered stations save in Cell 8

In [ ]:
# Function to find k nearest stations for each employee using efficient BallTree algorithm
# This is much faster than calculating distances to all stations individually
from sklearn.neighbors import BallTree
import numpy as np

def find_k_nearest_stations(locations_df, stations_df, k=5):
    """Find k nearest stations for each location using BallTree for efficient search"""
    
    # Convert coordinates to radians for haversine distance calculation
    # This gives us great-circle distances in kilometers
    locations_rad = np.radians(locations_df[['snapped_latitude', 'snapped_longitude']].values)
    stations_rad = np.radians(stations_df[['latitude', 'longitude']].values)
    
    # Build BallTree with haversine metric
    tree = BallTree(stations_rad, metric='haversine')
    
    # Find k nearest stations for each location
    distances, indices = tree.query(locations_rad, k=k)
    
    # Convert distances from radians to kilometers
    distances_km = distances * 6371  # Earth's radius in km
    
    results = []
    for i, (location_distances, location_indices) in enumerate(zip(distances_km, indices)):
        for j, (dist, idx) in enumerate(zip(location_distances, location_indices)):
            station = stations_df.iloc[idx]
            results.append({
                'location_index': i,
                'employee_id': locations_df.iloc[i]['employee_id'] if 'employee_id' in locations_df.columns else None,
                'candidate_rank': j + 1,
                'station_id': station['station_id'],
                'station_name': station['name'],
                'station_latitude': station['latitude'],
                'station_longitude': station['longitude'],
                'transport_category': station['transport_category'],
                'straight_line_distance_km': dist
            })
    
    return pd.DataFrame(results)

# Find 5 nearest stations for each employee
print("Finding 5 nearest stations for each employee...")
employee_candidates = find_k_nearest_stations(employee_df, hvv_stations_filtered, k=5)

print(f"Found {len(employee_candidates)} candidate stations for {len(employee_df)} employees")
print(f"Average stations per employee: {len(employee_candidates)/len(employee_df):.1f}")

# Show sample results
print(employee_candidates.head(10))

In [ ]:
# Function to calculate walking distance between two points using OpenRouteService API
# This gives us realistic walking times instead of just straight-line distances
import time
import os
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

def calculate_walking_distance(origin_lat, origin_lon, dest_lat, dest_lon, api_key, max_retries=5):
    """Calculate walking distance with robust retry logic for rate limiting"""
    
    url = "https://api.openrouteservice.org/v2/directions/foot-walking"
    
    headers = {
        'Authorization': api_key,
        'Content-Type': 'application/json',
        'Accept': 'application/json'
    }
    
    body = {
        'coordinates': [[origin_lon, origin_lat], [dest_lon, dest_lat]]
    }
    
    # Try multiple times in case of rate limiting or temporary errors
    for attempt in range(max_retries):
        try:
            response = requests.post(url, json=body, headers=headers)
            
            if response.status_code == 429:
                # Rate limited - wait longer with exponential backoff
                wait_time = min(10, 2 ** attempt)  # Cap at 10 seconds
                print(f"  Rate limited, waiting {wait_time}s...")
                time.sleep(wait_time)
                continue
                
            if response.status_code == 403:
                # Forbidden - likely quota exceeded
                print(f"  API quota exceeded (403), waiting 30s...")
                time.sleep(30)
                continue
                
            response.raise_for_status()
            data = response.json()
            
            if 'routes' in data and len(data['routes']) > 0:
                route = data['routes'][0]
                distance_m = route['summary']['distance']
                duration_s = route['summary']['duration']
                return distance_m, duration_s
            else:
                return None, None
                
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"  Error after {max_retries} attempts: {e}")
                return None, None
            time.sleep(2)  # Wait 2 seconds on error
    
    return None, None


# Now let's calculate walking distances for all the candidate stations we found
print("Calculating walking distances for candidate stations...")

# Add walking columns to candidate stations dataframe
employee_candidates['walking_distance_m'] = None
employee_candidates['walking_time_min'] = None

# Cache file for walking distances to avoid recalculating
walking_cache_file = 'data/candidate_walking_distances_cachedata/data/.csv'

# Load cache if exists to avoid recalculating previously computed distances
if os.path.exists(walking_cache_file):
    print("Loading cached walking distances...")
    walking_cache = pd.read_csv(walking_cache_file)
    
    # Merge cached results
    for idx, row in walking_cache.iterrows():
        mask = (employee_candidates['employee_id'] == row['employee_id']) & \
               (employee_candidates['station_id'] == row['station_id'])
        employee_candidates.loc[mask, 'walking_distance_m'] = row['walking_distance_m']
        employee_candidates.loc[mask, 'walking_time_min'] = row['walking_time_min']
    
    already_calculated = employee_candidates['walking_distance_m'].notna().sum()
    print(f"Loaded {already_calculated} cached walking distances")

# Calculate missing walking distances
to_calculate = employee_candidates[employee_candidates['walking_distance_m'].isna()]

print(f"Need to calculate walking distances for {len(to_calculate)} candidate stations")

processed_count = 0
failed_count = 0

# Calculate walking distances for each candidate station
for idx, row in to_calculate.iterrows():
    print(f"Processing {row['employee_id']} - {row['station_name']} (rank {row['candidate_rank']}) ({processed_count + 1}/{len(to_calculate)})...")
    
    try:
        # Get employee coordinates from the original dataframe
        employee_data = employee_df[employee_df['employee_id'] == row['employee_id']].iloc[0]
        
        dist_m, time_s = calculate_walking_distance(
            employee_data['snapped_latitude'], employee_data['snapped_longitude'],
            row['station_latitude'], row['station_longitude'],
            ORS_API_KEY
        )
        
        if dist_m is not None and time_s is not None:
            employee_candidates.loc[idx, 'walking_distance_m'] = dist_m
            employee_candidates.loc[idx, 'walking_time_min'] = time_s / 60
            print(f"  Distance: {dist_m}m, Time: {time_s/60:.1f}min")
            processed_count += 1
        else:
            print(f"  Failed to get walking distance - using geographic distance as fallback")
            # Use geographic distance as fallback
            geo_dist_km = row['straight_line_distance_km']
            # Estimate walking time (assuming 5 km/h = 12 min/km)
            est_walk_time_min = geo_dist_km * 12
            employee_candidates.loc[idx, 'walking_distance_m'] = geo_dist_km * 1000
            employee_candidates.loc[idx, 'walking_time_min'] = est_walk_time_min
            print(f"  Using geographic distance: {geo_dist_km:.3f}km, estimated time: {est_walk_time_min:.1f}min")
            processed_count += 1
            
    except Exception as e:
        print(f"  Error: {e} - using geographic distance as fallback")
        # Use geographic distance as fallback
        geo_dist_km = row['straight_line_distance_km']
        est_walk_time_min = geo_dist_km * 12
        employee_candidates.loc[idx, 'walking_distance_m'] = geo_dist_km * 1000
        employee_candidates.loc[idx, 'walking_time_min'] = est_walk_time_min
        processed_count += 1
    
    # Rate limiting to avoid API quota issues
    time.sleep(1.5)
    
    # Save progress frequently so we don't lose work if something fails
    if processed_count % 5 == 0:
        employee_candidates.to_csv(walking_cache_file, index=False)
        print(f"Saved progress: {processed_count} calculated")

# Final save
employee_candidates.to_csv(walking_cache_file, index=False)
print(f"\nCompleted: {processed_count} calculated, {failed_count} failed")

# Save final candidate stations with walking times
employee_candidates.to_csv("data/employee_candidate_stations_with_walkingdata/data/.csv", index=False)
print("Saved candidate stations with walking distances to data/employee_candidate_stations_with_walkingdata/data/.csv")

# Display sample results
print("\nSample candidate stations with walking times:")
print(employee_candidates[
    ['employee_id', 'station_name', 'candidate_rank', 'straight_line_distance_km', 'walking_distance_m', 'walking_time_min']
].head(20))

In [ ]:
# Now let's find candidate stations for the J&J office using the same approach
print("Finding candidate stations for J&J office (k=5)...")

# Create a simple dataframe for the office location
office_df = pd.DataFrame([{
    'employee_id': 'JJ_OFFICE',
    'snapped_latitude': JJ_OFFICE['snapped_latitude'],
    'snapped_longitude': JJ_OFFICE['snapped_longitude']
}])

# Find 5 nearest stations for the office
office_candidates = find_k_nearest_stations(office_df, hvv_stations_filtered, k=5)

print(f"Found {len(office_candidates)} candidate stations for office")

# Rename the location_index to something more meaningful
office_candidates['location_type'] = 'office'

print("\nOffice candidate stations:")
print(office_candidates)

# Save office candidates
office_candidates.to_csv("data/jj_office_candidate_stations.csv", index=False)
print("Saved office candidate stations to data/jj_office_candidate_stations.csv")

# Calculate walking distances for office candidates
print("\nCalculating walking distances for office candidate stations...")

office_candidates['walking_distance_m'] = None
office_candidates['walking_time_min'] = None

for idx, row in office_candidates.iterrows():
    print(f"Processing {row['station_name']} (rank {row['candidate_rank']})...")
    
    try:
        dist_m, time_s = calculate_walking_distance(
            row['station_latitude'], row['station_longitude'],
            JJ_OFFICE['snapped_latitude'], JJ_OFFICE['snapped_longitude'],
            ORS_API_KEY
        )
        
        if dist_m is not None and time_s is not None:
            office_candidates.loc[idx, 'walking_distance_m'] = dist_m
            office_candidates.loc[idx, 'walking_time_min'] = time_s / 60
            print(f"  Distance: {dist_m}m, Time: {time_s/60:.1f}min")
        else:
            print(f"  Failed to get walking distance - using geographic distance as fallback")
            geo_dist_km = row['straight_line_distance_km']
            est_walk_time_min = geo_dist_km * 12
            office_candidates.loc[idx, 'walking_distance_m'] = geo_dist_km * 1000
            office_candidates.loc[idx, 'walking_time_min'] = est_walk_time_min
            print(f"  Using geographic distance: {geo_dist_km:.3f}km, estimated time: {est_walk_time_min:.1f}min")
            
    except Exception as e:
        print(f"  Error: {e} - using geographic distance as fallback")
        geo_dist_km = row['straight_line_distance_km']
        est_walk_time_min = geo_dist_km * 12
        office_candidates.loc[idx, 'walking_distance_m'] = geo_dist_km * 1000
        office_candidates.loc[idx, 'walking_time_min'] = est_walk_time_min
    
    time.sleep(0.5)

# Save office candidates with walking times
office_candidates.to_csv("data/jj_office_candidate_stations_with_walkingdata/data/.csv", index=False)
print("Saved office candidate stations with walking distances to data/jj_office_candidate_stations_with_walkingdata/data/.csv")

print("\nOffice candidate stations with walking times:")
print(office_candidates[
    ['station_name', 'candidate_rank', 'straight_line_distance_km', 'walking_distance_m', 'walking_time_min']
])

In [13]:
# Skip - Old single-station approach (replaced by multi-candidate approach in Cells 10-12)

In [14]:
# Skip - Old single-station approach (replaced by multi-candidate approach in Cells 10-12)

In [16]:
# Create a summary of J&J office transport access using multi-candidate approach
office_access_summary = {
    'office_address': JJ_OFFICE['address'],
    'office_latitude': JJ_OFFICE['latitude'],
    'office_longitude': JJ_OFFICE['longitude'],
    'office_station_name': office_candidates_df.iloc[0]['station_name'],  # Best candidate
    'office_station_id': office_candidates_df.iloc[0]['station_id'],
    'office_station_lat': office_candidates_df.iloc[0]['station_lat'],
    'office_station_lon': office_candidates_df.iloc[0]['station_lon'],
    'distance_to_station_km': office_candidates_df.iloc[0]['distance_km'],
    'walking_distance_m': office_candidates_df.iloc[0]['walking_distance_m'],
    'walking_time_min': office_candidates_df.iloc[0]['walking_time_min'],
    'station_type': office_candidates_df.iloc[0]['station_type']
}

# Convert to DataFrame and save
office_access_df = pd.DataFrame([office_access_summary])
office_access_df.to_csv('jj_office_station_access.csv', index=False)

print("J&J Office station access saved to jj_office_station_access.csv")

office_access_df

J&J Office station access saved to jj_office_station_access.csv


,office_address,office_latitude,office_longitude,office_station_name,office_station_id,office_station_lat,office_station_lon,distance_to_station_km,walking_distance_m,walking_time_min,station_type
0,"Robert-Koch-Straße 1, 22851 Norderstedt",53.6876,10.0053,"Harksheide, Libellengrund",12461266487,53.687363,10.008928,0.211641,308.9,3.706667,bus
